In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
import json
import os

In [2]:
# Dataset paths
TRAIN_DIR = "../dataset/train"
VALID_DIR = "../dataset/valid"
TEST_DIR = "../dataset/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dataset = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

valid_dataset = image_dataset_from_directory(
    VALID_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_dataset = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_dataset.class_names
print("Classes:", len(class_names))
print(class_names)

Found 37997 files belonging to 38 classes.
Found 8129 files belonging to 38 classes.
Found 8179 files belonging to 38 classes.
Classes: 38
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomat

In [3]:
# Apply EfficientNet preprocessing
train_dataset = train_dataset.map(
    lambda x, y: (tf.keras.applications.efficientnet.preprocess_input(x), y)
)

valid_dataset = valid_dataset.map(
    lambda x, y: (tf.keras.applications.efficientnet.preprocess_input(x), y)
)

test_dataset = test_dataset.map(
    lambda x, y: (tf.keras.applications.efficientnet.preprocess_input(x), y)
)

print("✅ Preprocessing applied")

✅ Preprocessing applied


In [4]:
# Load EfficientNetB0
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
])

# Build the model
model.build((None, 224, 224, 3))

# Show summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional)  (None, 7, 7, 1280)       4049571   
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 256)               327936    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 38)                9766      
                                                        

In [5]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("✅ Model compiled")

✅ Model compiled


In [7]:
model.save("../models/crop_disease_model.keras")
print("✅ Model saved successfully")

TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.

In [6]:
import traceback

try:
    config = model.get_config()

    def find_tensor(obj, path="root"):
        import tensorflow as tf

        if isinstance(obj, tf.Tensor):
            print("FOUND TENSOR:", path)
            print(obj)
            return True

        if isinstance(obj, dict):
            for k, v in obj.items():
                if find_tensor(v, f"{path}.{k}"):
                    return True

        elif isinstance(obj, list):
            for i, v in enumerate(obj):
                if find_tensor(v, f"{path}[{i}]"):
                    return True

        return False

    find_tensor(config)

except Exception:
    traceback.print_exc()
    

FOUND TENSOR: root.layers[1].config.layers[3].config.scale
tf.Tensor([2.0896919 2.1128857 2.1081853], shape=(3,), dtype=float32)


In [6]:
print(type(class_names))
print(class_names)

<class 'list'>
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tom

In [7]:
print(type(class_names[0]))

<class 'str'>


In [8]:
print(model.get_config())

{'name': 'sequential', 'layers': [{'class_name': 'InputLayer', 'config': {'batch_input_shape': (None, 224, 224, 3), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'efficientnetb0_input'}}, {'class_name': 'Functional', 'config': {'name': 'efficientnetb0', 'layers': [{'class_name': 'InputLayer', 'config': {'batch_input_shape': (None, 224, 224, 3), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_1'}, 'name': 'input_1', 'inbound_nodes': []}, {'class_name': 'Rescaling', 'config': {'name': 'rescaling', 'trainable': False, 'dtype': 'float32', 'scale': 0.00392156862745098, 'offset': 0.0}, 'name': 'rescaling', 'inbound_nodes': [[['input_1', 0, 0, {}]]]}, {'class_name': 'Normalization', 'config': {'name': 'normalization', 'trainable': False, 'dtype': 'float32', 'axis': (3,), 'mean': None, 'variance': None}, 'name': 'normalization', 'inbound_nodes': [[['rescaling', 0, 0, {}]]]}, {'class_name': 'Rescaling', 'config': {'name': 'rescaling_1', 'trainable': False, '

In [9]:
config = model.get_config()
print("Config generated successfully")

Config generated successfully


In [10]:
print(type(model.optimizer))
print(type(model.loss))
print(model.loss)
print(model.metrics)

<class 'keras.optimizers.optimizer_v2.adam.Adam'>
<class 'str'>
categorical_crossentropy
[<keras.metrics.base_metric.Mean object at 0x000002AF5A75F940>, <keras.metrics.base_metric.MeanMetricWrapper object at 0x000002AF668BBBB0>]


In [11]:
print(tf.__version__)
print(tf.keras.__version__)

2.10.0
2.10.0


In [12]:
import json

config = model.get_config()

json.dumps(config)

print("JSON serialization successful")

TypeError: Object of type EagerTensor is not JSON serializable

In [7]:
checkpoint = ModelCheckpoint(
    "../models/crop_disease_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

print("✅ Callbacks ready")

✅ Callbacks ready


In [6]:
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=5
)

Epoch 1/5
1188/1188 [==============================] - 1301s 1s/step - loss: 0.4990 - accuracy: 0.8536 - val_loss: 0.1404 - val_accuracy: 0.9584
Epoch 2/5
1188/1188 [==============================] - 1167s 982ms/step - loss: 0.2117 - accuracy: 0.9293 - val_loss: 0.1071 - val_accuracy: 0.9689
Epoch 3/5
1188/1188 [==============================] - 1086s 914ms/step - loss: 0.1727 - accuracy: 0.9420 - val_loss: 0.0947 - val_accuracy: 0.9675
Epoch 4/5
1188/1188 [==============================] - 1047s 881ms/step - loss: 0.1517 - accuracy: 0.9492 - val_loss: 0.0835 - val_accuracy: 0.9734
Epoch 5/5
1188/1188 [==============================] - 1021s 859ms/step - loss: 0.1433 - accuracy: 0.9508 - val_loss: 0.0873 - val_accuracy: 0.9708


In [7]:
import tensorflow as tf

tf.saved_model.save(model, "../models/crop_disease_saved")
print("SavedModel saved successfully")

INFO:tensorflow:Assets written to: ../models/crop_disease_saved\assets


INFO:tensorflow:Assets written to: ../models/crop_disease_saved\assets


SavedModel saved successfully


In [8]:
import json

with open("../models/class_names.json", "w") as f:
    json.dump(class_names, f)

print("class_names.json saved")

class_names.json saved


In [9]:
import os

print(os.listdir("../models/crop_disease_saved"))

['assets', 'saved_model.pb', 'variables']


In [12]:
import json

with open("../models/class_names.json", "w") as f:
    json.dump(class_names, f)

print("✅ class_names.json saved")

✅ class_names.json saved


In [13]:
test_loss, test_acc = model.evaluate(test_dataset)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

256/256 [==============================] - 177s 689ms/step - loss: 0.0881 - accuracy: 0.9727
Test Accuracy: 0.9727350473403931
Test Loss: 0.08806674927473068


In [14]:
import os

os.makedirs("../models", exist_ok=True)

model.save("../models/crop_disease_model.keras")

print("✅ Model saved successfully!")


TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.

In [15]:
print(type(model))


<class 'keras.engine.sequential.Sequential'>


In [16]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional)  (None, 7, 7, 1280)       4049571   
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 256)               327936    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 38)                9766      
                                                        

In [17]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.10.0
Keras: 2.10.0


In [19]:
print(model.optimizer)

In [10]:
model.save("../models/crop_disease_saved.keras")

TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.

In [11]:
print(type(model))

<class 'keras.engine.sequential.Sequential'>


In [12]:
new_model = tf.keras.models.clone_model(model)

In [13]:
new_model.build((None, 224, 224, 3))

In [14]:
new_model.set_weights(model.get_weights())

In [15]:
new_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
new_model.save("../models/crop_disease_fixed.keras")

TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.

In [17]:
import tensorflow as tf

loaded = tf.saved_model.load("../models/crop_disease_saved")
print(loaded.signatures)

_SignatureMap({'serving_default': <ConcreteFunction signature_wrapper(*, efficientnetb0_input) at 0x21823E785E0>})


In [20]:
print(model.loss)

categorical_crossentropy


In [21]:
print(model.metrics)

[<keras.metrics.base_metric.Mean object at 0x00000177EDB2BEE0>, <keras.metrics.base_metric.MeanMetricWrapper object at 0x00000177F067C220>]


In [22]:
import tensorflow as tf

tf.saved_model.save(model, "../models/crop_disease_saved")
print("Saved successfully!")

INFO:tensorflow:Assets written to: ../models/crop_disease_saved\assets


INFO:tensorflow:Assets written to: ../models/crop_disease_saved\assets


Saved successfully!


In [23]:
import os

print(os.path.getsize("../models/crop_disease_model.keras"))

6144


In [24]:
model.to_json()


TypeError: Unable to serialize [2.0896919 2.1128857 2.1081853] to JSON. Unrecognized type <class 'tensorflow.python.framework.ops.EagerTensor'>.